# 🧩 Mini-Lab: The Project Folder Template

**Module 4: Escape the Notebook: Environment, Repo, Git** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Identify** what belongs in `src/`, `tests/`, `data/`, and `docs/`
2. **Move** a flat v1.5 project into that template as an importable `triage` package
3. **Run** the package from the project root with `python -m triage`

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Modules, packages, imports | Files (modules) and folders with __init__.py (packages) make code importable from anywhere, so your pipeline logic is reusable, testable, and easier to run outside notebooks. |
| Module entry point | `__main__.py` plus `if __name__ == "__main__"` make `python -m triage` run the package without executing it on import |


## Why Restructure?

Recall from **mini-build-triage-package**: a package is a folder with `__init__.py`, and you import it like any library.

That demo lived next to the notebook as a bare `triage/` folder. Fine for a lesson. A stranger cloning your repo still has to guess:

- Where is the code?
- Where are the tests?
- Where is the sample data?
- How do I *run* this without opening a notebook?

This mini-lab takes a messy **v1.5** (everything dumped in one folder) and moves it into the template the rest of the course uses: `src/`, `tests/`, `data/`, `docs/`. Then we add an entry point so `python -m triage` works.


## The Messy v1.5 Layout

Here is a tiny support-triage project as it often looks after the first few modules: one folder, mixed files, no package, no way to run it as a module.

```
v15_project/
├── classify.py         # the actual code
├── test_classify.py    # a test sitting next to production code
├── messages.csv        # data mixed in with Python files
└── notes.md            # docs mixed in too
```

We will create that layout, inspect it, then move it.


In [3]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


ROOT = Path("v15_project")
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir()

(ROOT / "classify.py").write_text(
    '''"""Keyword classifier for support messages."""


def classify_message(text: str) -> dict:
    """Return a label dict for a single support message."""
    lowered = text.lower()
    if "invoice" in lowered or "refund" in lowered or "charge" in lowered:
        label = "billing"
    elif "login" in lowered or "password" in lowered:
        label = "account"
    else:
        label = "general"
    return {"label": label, "text": text}
'''.lstrip(),
    encoding="utf-8",
)

(ROOT / "test_classify.py").write_text(
    '''from classify import classify_message


def test_billing():
    result = classify_message("Please refund last month's charge")
    assert result["label"] == "billing"
'''.lstrip(),
    encoding="utf-8",
)

(ROOT / "messages.csv").write_text(
    "ticket_id,text\n"
    "1,Please refund last month's charge\n"
    "2,I forgot my password\n",
    encoding="utf-8",
)

(ROOT / "notes.md").write_text(
    "v1.5 notes: keyword classifier only. No model call yet.\n",
    encoding="utf-8",
)

files = [p.relative_to(ROOT).as_posix() for p in sorted(ROOT.rglob("*")) if p.is_file()]
md("**v1.5 files (flat):** `" + "`, `".join(files) + "`")


**v1.5 files (flat):** `classify.py`, `messages.csv`, `notes.md`, `test_classify.py`

## The Template: `src`, `tests`, `data`, `docs`

Each top-level folder answers one question a newcomer asks:

| Folder | Holds | Newcomer question |
|--------|-------|-------------------|
| `src/` | Importable Python package | Where is the code I run and import? |
| `tests/` | Tests that import the package | How do I check it still works? |
| `data/` | Sample CSVs and fixtures | What input can I try? |
| `docs/` | Notes, not executable code | Where do I read about it? |

The package itself still follows the M2 rule: a folder named `triage` with `__init__.py`. It just lives under `src/` instead of the repo root:

```
v15_project/
├── src/
│   └── triage/
│       ├── __init__.py
│       ├── __main__.py      ← makes `python -m triage` work
│       └── classify.py
├── tests/
│   └── test_classify.py
├── data/
│   └── messages.csv
└── docs/
    └── notes.md
```


In [4]:
# Move v1.5 into the template. Same functions, new locations.

src_pkg = ROOT / "src" / "triage"
tests = ROOT / "tests"
data = ROOT / "data"
docs = ROOT / "docs"
for folder in (src_pkg, tests, data, docs):
    folder.mkdir(parents=True, exist_ok=True)

(src_pkg / "__init__.py").write_text(
    '''"""Support-message triage package."""

from .classify import classify_message

__all__ = ["classify_message"]
'''.lstrip(),
    encoding="utf-8",
)

shutil.move(str(ROOT / "classify.py"), str(src_pkg / "classify.py"))

(src_pkg / "__main__.py").write_text(
    '''"""Run the package: python -m triage"""

from .classify import classify_message


def main() -> None:
    result = classify_message("Please refund last month's charge")
    print(result)


if __name__ == "__main__":
    main()
'''.lstrip(),
    encoding="utf-8",
)

(tests / "test_classify.py").write_text(
    '''from triage.classify import classify_message


def test_billing():
    result = classify_message("Please refund last month's charge")
    assert result["label"] == "billing"
'''.lstrip(),
    encoding="utf-8",
)
(ROOT / "test_classify.py").unlink()

shutil.move(str(ROOT / "messages.csv"), str(data / "messages.csv"))
shutil.move(str(ROOT / "notes.md"), str(docs / "notes.md"))


def tree_lines(path: Path, prefix: str = "") -> list[str]:
    entries = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name))
    lines = []
    for i, entry in enumerate(entries):
        last = i == len(entries) - 1
        branch = "└── " if last else "├── "
        lines.append(f"{prefix}{branch}{entry.name}")
        if entry.is_dir():
            extension = "    " if last else "│   "
            lines.extend(tree_lines(entry, prefix + extension))
    return lines


listing = "\n".join([f"{ROOT.name}/"] + tree_lines(ROOT))
md(f"**After the move:**\n\n```\n{listing}\n```")


**After the move:**

```
v15_project/
├── data
│   └── messages.csv
├── docs
│   └── notes.md
├── src
│   └── triage
│       ├── __init__.py
│       ├── __main__.py
│       └── classify.py
└── tests
    └── test_classify.py
```

## Make It Runnable: `__main__.py` and `if __name__ == "__main__"`

Two small files turn a package you can *import* into a package you can *run*:

1. **`src/triage/__main__.py`** — Python looks for this file when you type `python -m triage`.
2. **`if __name__ == "__main__":`** — the `main()` call runs only when the file is the entry point, not when someone does `from triage import classify_message`.

Because the package lives under `src/`, the project root is not on `sys.path` by default. Point `PYTHONPATH` at `src/` (or later, `pip install -e .`). Then, from the project root:

```bash
# macOS / Linux
PYTHONPATH=src python -m triage

# Windows PowerShell
$env:PYTHONPATH = "src"
python -m triage
```

The next cell does the same thing with `subprocess` so it works on every OS from this notebook.


In [5]:
env = os.environ.copy()
src = str((ROOT / "src").resolve())
existing = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = src if not existing else src + os.pathsep + existing

completed = subprocess.run(
    [sys.executable, "-m", "triage"],
    cwd=str(ROOT.resolve()),
    env=env,
    capture_output=True,
    text=True,
    encoding="utf-8",
)
print("exit code:", completed.returncode)
print("stdout:", completed.stdout.strip())
if completed.stderr.strip():
    print("stderr:", completed.stderr.strip())


exit code: 0
stdout: {'label': 'billing', 'text': "Please refund last month's charge"}


`python -m triage` printed a result. Importing the same package should **not** print anything — that is the point of the `if __name__ == "__main__"` guard.


In [6]:
# Import the package the same way a test or notebook would:
# put src/ on sys.path, then import triage.
src_path = str((ROOT / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from triage import classify_message

imported = classify_message("Please refund last month's charge")
print("imported result:", imported)
print("same function, no extra print from main()")


imported result: {'label': 'billing', 'text': "Please refund last month's charge"}
same function, no extra print from main()


In [7]:
print(classify_message("I forgot my password"))

{'label': 'account', 'text': 'I forgot my password'}


The import returned a dict and printed nothing extra. Same function as `python -m triage` — the guard kept `main()` from running on import.

In the terminal, a stranger standing in the project root runs:

```bash
PYTHONPATH=src python -m triage          # macOS / Linux
# $env:PYTHONPATH = "src"; python -m triage   # Windows PowerShell
```


In [8]:
cheat_sheet = """
### Project Folder Template — Quick Reference

| Piece | Role |
|-------|------|
| `src/triage/` | The importable package (code the product runs) |
| `src/triage/__init__.py` | Marks the folder as a package; re-exports public names |
| `src/triage/__main__.py` | Entry point for `python -m triage` |
| `if __name__ == "__main__"` | Runs `main()` only as an entry point, never on import |
| `tests/` | Tests import `triage`, they do not sit next to production files |
| `data/` | Sample inputs (CSV, JSON) kept out of `src/` |
| `docs/` | Human-readable notes, not executed by Python |
| `PYTHONPATH=src` | Makes `import triage` and `python -m triage` work before you add an editable install |
"""
md(cheat_sheet)



### Project Folder Template — Quick Reference

| Piece | Role |
|-------|------|
| `src/triage/` | The importable package (code the product runs) |
| `src/triage/__init__.py` | Marks the folder as a package; re-exports public names |
| `src/triage/__main__.py` | Entry point for `python -m triage` |
| `if __name__ == "__main__"` | Runs `main()` only as an entry point, never on import |
| `tests/` | Tests import `triage`, they do not sit next to production files |
| `data/` | Sample inputs (CSV, JSON) kept out of `src/` |
| `docs/` | Human-readable notes, not executed by Python |
| `PYTHONPATH=src` | Makes `import triage` and `python -m triage` work before you add an editable install |


## Cleanup

Remove the demo project folder so it does not clutter the lab directory. The layout above is the template to use on your real repo.


In [9]:
shutil.rmtree(ROOT, ignore_errors=True)
print(f"Cleaned up {ROOT.as_posix()}")


Cleaned up v15_project


## 🎯 Summary

### Key Takeaways

1. **`src/`, `tests/`, `data/`, `docs/`** — a stranger can find code, tests, sample input, and notes without asking you
2. **Package under `src/`** — `triage/` still needs `__init__.py`; the `src/` parent keeps that package off the repo root
3. **`python -m triage`** — `__main__.py` is the file Python runs when you invoke the package as a module
4. **`if __name__ == "__main__"`** — `main()` runs from the command line and stays quiet when the package is imported
